# TypeScript 与工具链

学习目标：能使用项目内编译器检查 TypeScript、生成并运行 JavaScript，区分类型诊断与运行时异常。

前置知识：JavaScript 变量、函数、字符串与函数调用；会在终端运行 Node.js 文件。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/01-typescript-toolchain/。

1. [greeting.ts](scripts/01-typescript-toolchain/greeting.ts)：带类型标注的正常程序。
2. [type-error.ts](scripts/01-typescript-toolchain/type-error.ts)、[runtime-error.ts](scripts/01-typescript-toolchain/runtime-error.ts)：类型错误和运行时异常。
3. [tsconfig.json](scripts/01-typescript-toolchain/tsconfig.json)、[tsconfig.errors.json](scripts/01-typescript-toolchain/tsconfig.errors.json)：分别检查正常程序和预期类型错误。

工具版本与命令入口位于本技术目录的 [package.json](package.json)，依赖锁定在 [package-lock.json](package-lock.json)。

## 1 在 JavaScript 上增加类型信息

TypeScript 在 JavaScript 的基础上增加类型语法和静态类型检查。下面 name 后的 : string 表示这个形参需要字符串；函数调用仍使用熟悉的 JavaScript 写法。

```typescript
function greet(name: string) {
  return "你好，" + name;
}
console.log(greet("小林")); // 你好，小林
```

配套文件：[greeting.ts](scripts/01-typescript-toolchain/greeting.ts)。

静态检查发生在执行程序之前，帮助发现类型不匹配。这里的类型标注不会在运行时检查传入数据，也不会把其他值自动转换为字符串。

## 2 使用项目内工具

- Node.js 负责执行生成的 JavaScript。
- npm 根据依赖清单安装工具，并执行 package.json 中的命令。
- TypeScript 编译器 tsc 负责类型检查和代码生成；@types/node 提供 Node.js API 的类型声明，不提供这些 API 的运行实现。

按本目录 README 完成依赖安装后，检查类型：

```bash
npm run check:01
# 无诊断并以状态 0 退出，表示本章正常项目通过类型检查。
```

Windows PowerShell 若限制 npm.ps1，可以将命令中的 npm 换为 npm.cmd。npm run 会找到本项目安装的编译器，无需全局安装 TypeScript。

本章的检查入口等价于执行项目内 tsc -p scripts/01-typescript-toolchain --noEmit；-p 明确指定项目配置，--noEmit 表示只检查，不写出 JavaScript。

## 3 阅读最小项目配置

[tsconfig.json](scripts/01-typescript-toolchain/tsconfig.json) 规定编译选项和入口文件：

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "strict": true,
    "lib": ["ES2025"],
    "types": ["node"],
    "rootDir": ".",
    "outDir": "../../.build/01-typescript-toolchain",
    "noEmitOnError": true
  },
  "files": ["greeting.ts", "runtime-error.ts"]
}
```

- strict 开启一组严格类型检查；学习和修改示例时保持开启。
- target 指定输出使用的 JavaScript 语法目标；lib 选择标准库类型声明。二者都不会给运行环境安装缺失的 API。
- module 为 NodeNext，并结合本技术目录 package.json 的 "type": "module"，让本例按 Node.js 的 ES 模块规则检查和输出。
- types 明确载入 node 的类型声明，使编译器认识 console 等 Node.js API。
- rootDir 和 outDir 决定源码与输出目录；这些路径相对于当前配置文件。这里的 ../../.build 回到本技术目录下的 .build。
- files 列出本项目的入口源码；被源码导入的文件也可能进入项目。noEmitOnError 让出现编译错误时不生成新输出。

本章配置只包含 greeting.ts 和 runtime-error.ts，类型错误文件由另一份配置单独检查。

## 4 检查、生成与运行

先明确三个命令的职责，再按顺序执行。下面把同一份源码在检查阶段与运行阶段的去向画开。

check:01 用于确认类型契约，build:01 产生本次源码的 JavaScript，run:01 才执行生成物。先检查再生成、最后运行，才能避免旧产物混入本轮观察。

![检查、生成、运行分别完成什么。三个 npm 入口是本章命令约定；检查通过不会自动运行程序。](image/illustration/01-01-check-build-run.svg)

图示说明：依据本章命令与类型擦除规则自绘，图只展示本章变量、函数标注的转换；其他 TypeScript 语法可能需要额外生成代码。

执行 Step 1 后不要期待输出文件；执行 Step 2 后检查生成物，再在 Step 3 观察问候语。

Step 1：只检查类型。

```bash
npm run check:01
# 执行 tsc -p scripts/01-typescript-toolchain --noEmit。
```

Step 2：检查并生成 JavaScript。

```bash
npm run build:01
# 输出到 .build/01-typescript-toolchain/。
```

Step 3：运行生成的示例。

```bash
npm run run:01
# 运行 .build/01-typescript-toolchain/greeting.js，输出 你好，小林。
```

check:01、build:01、run:01 是 package.json 中为命令取的入口名称，不是 TypeScript 关键字。build:01 成功后再运行 run:01；输出目录被清理后需要重新生成。

## 5 类型擦除与输出代码

类型擦除（type erasure）指仅用于检查的类型信息不会保留在运行代码中。例如 : string 在生成的 greeting.js 中消失。下面是该文件中的函数和调用部分：

```javascript
function greet(name) {
    return "你好，" + name;
}
console.log(greet("小林")); // 你好，小林
```

本例输出末尾还包含用于保持模块身份的 export {}。这里先关注参数标注的消失，模块语法在后续章节展开。

类型擦除不表示所有 TypeScript 语法都能直接删掉；有些语法需要生成 JavaScript。本章先使用变量与函数标注。Node.js 能直接运行某些 .ts 文件，也不代表执行了 tsc 的静态检查。

## 6 阅读类型诊断

把字符串赋给要求数值的变量，编译器会指出赋值关系不成立：

```typescript
const price: number = "19"; // TS2322：string 不能赋给 number
console.log(price); // 本例只做类型检查；报 TS2322，不生成或运行这条输出。
```

配套 [type-error.ts](scripts/01-typescript-toolchain/type-error.ts) 使用独立的 [tsconfig.errors.json](scripts/01-typescript-toolchain/tsconfig.errors.json)：

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": { "noEmit": true },
  "files": ["type-error.ts"]
}
```

extends 继承正常配置，files 改为错误文件，noEmit 确保只检查。执行时预期失败：

```bash
npm run errors:01
# 预期非零退出；type-error.ts 第 5 行报告 TS2322。
```

诊断先看文件与行列位置，再看错误代码和相关类型。这里的修正是让初始值符合需求，例如把 "19" 改为数值 19；不需要关闭 strict。

## 7 类型检查通过仍可能运行失败

JSON.parse() 解析 JSON 文本，其参数允许字符串。编译器不能仅凭 string 类型确认文本符合 JSON 格式。

```typescript
JSON.parse("{"); // 类型检查通过；运行时抛出 SyntaxError，因为 JSON 文本不完整
```

在已完成 build:01 后单独运行 [runtime-error.ts](scripts/01-typescript-toolchain/runtime-error.ts) 的输出文件：

```bash
node .build/01-typescript-toolchain/runtime-error.js
# 预期非零退出并出现 SyntaxError；具体措辞由 Node.js 决定。
```

这是运行时异常，不是编译器的 TS2322。类型检查不会替代 JSON 解析；此处直接保留解析器抛出的原始异常。

## 8 编辑器提示与命令行

编辑器可以在悬停时显示 name 的类型，并在类型错误处显示诊断。编辑器采用的 TypeScript 版本必须与项目匹配，否则可能与命令行给出不同结果。

TypeScript 7 使用新的原生实现。按 [VS Code 编辑器配置](README.md#vs-code-编辑器配置)安装并启用 TypeScript 7 扩展，选择本项目的编译器，再用本章 npm 命令对照诊断。仅安装 npm 依赖不会自动切换编辑器的语言服务。

## 本章小结

- 类型检查、生成 JavaScript、执行程序是三个步骤。
- 项目配置和锁文件固定编译条件；编辑器也要使用匹配版本。
- 类型标注不会自动转换或校验运行时数据，检查通过仍需观察程序行为。

## 练习

1. 将 greeting.ts 的名字改为自己的名字，依次检查、生成、运行，确认输出与修改一致。
2. 把 type-error.ts 的初始值改为数值 19，再运行 errors:01，确认类型诊断消失；练习后恢复原例。
3. 将 runtime-error.ts 的参数改为合法 JSON 文本 "{}"，重新生成后运行，确认异常消失。说明为什么修改前后都能通过类型检查。

### 提示

1. 修改调用处的实参，保留 greet 的 string 参数。
2. errors:01 是命令名称，不承诺每次都必须失败；结果取决于这次源码。
3. 先重新生成 runtime-error.js，再运行，避免观察旧产物。


### 参考解析

1. 例如实参改为 小雨，预期输出 你好，小雨；检查通过本身不产生这行输出。
2. 改为数值 19 后不再有对应 TS2322，恢复字符串后应重新出现诊断。
3. 两个参数的静态类型都是 string；只有运行 JSON.parse 时才解析文本结构。改为合法文本后正常结束；原程序没有 console.log，因此不会自动显示解析对象。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [The Basics](https://www.typescriptlang.org/docs/handbook/2/basic-types.html) 中 Static type-checking、Emitting with Errors、Erased Types、Strictness；[strict](https://www.typescriptlang.org/tsconfig/strict.html)、[target](https://www.typescriptlang.org/tsconfig/target.html)、[lib](https://www.typescriptlang.org/tsconfig/lib.html)、[types](https://www.typescriptlang.org/tsconfig/types.html)、[files](https://www.typescriptlang.org/tsconfig/files.html)、[outDir](https://www.typescriptlang.org/tsconfig/outDir.html)、[noEmitOnError](https://www.typescriptlang.org/tsconfig/noEmitOnError.html)、[模块参考](https://www.typescriptlang.org/docs/handbook/modules/reference.html#node16-nodenext)：检查、生成与项目配置。 |
| Microsoft Developer Blogs | [Announcing TypeScript 7.0](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/)：原生编译器及编辑器支持条件。 |
| npm | [npm ci](https://docs.npmjs.com/cli/v11/commands/npm-ci/)、[npm run](https://docs.npmjs.com/cli/v11/commands/npm-run/)：锁文件安装与项目命令执行。 |
| Node.js 24.11.0 | [模块系统判定](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#determining-module-system)、[TypeScript](https://nodejs.org/download/release/v24.11.0/docs/api/typescript.html#type-stripping)：运行模块与类型剥离的检查边界。 |
| TC39 | [ECMAScript 2025 §25.5.1 JSON.parse](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-json.parse)：JSON 解析与格式错误的 SyntaxError。 |
